In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (f1_score, accuracy_score,precision_score, recall_score,roc_auc_score, confusion_matrix,classification_report)

In [2]:
df = pd.read_csv(r'C:\Infoct_project 1 sample practice\dataset\predictive_maintenance.csv')
print("Dataset loaded!")
print(df.shape)

Dataset loaded!
(10000, 10)


In [3]:
# Week 2 external context
np.random.seed(42)
df['Ambient_temperature'] = np.random.normal(loc=295, scale=3, size=len(df))
df['Load_density'] = np.random.uniform(low=0.3, high=1.0, size=len(df))

# Week 1 engineered features
df["temp_difference"] = df["Process temperature [K]"] - df["Air temperature [K]"]
df["power"] = df["Rotational speed [rpm]"] * df["Torque [Nm]"]
df["tool_wear_rate"] = df["Tool wear [min]"] / df["Rotational speed [rpm]"]
df["heat_stress_index"] = df["Air temperature [K]"] * df["Tool wear [min]"]
df["wear_per_rotation"] = df["Tool wear [min]"] / df["Rotational speed [rpm]"] * 1000

# Week 3 new features
df["thermal_stress"] = df["temp_difference"] * df["tool_wear_rate"]
df["power_per_temp"] = df["power"] / df["Air temperature [K]"]

print("All features recreated!")
print(df.shape)

All features recreated!
(10000, 19)


In [4]:
# Final 12 features
final_features = ['Air temperature [K]', 'Process temperature [K]','Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]','temp_difference', 'power', 'tool_wear_rate',
'heat_stress_index', 'wear_per_rotation','thermal_stress', 'power_per_temp']

X = df[final_features]
Y = df['Target']

X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42, stratify=Y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train LightGBM
model = lgb.LGBMClassifier(
    n_estimators=100,
    learning_rate=0.1,
    random_state=42,
    class_weight='balanced'
)
model.fit(X_train_scaled, Y_train)
print("Model trained successfully!")

[LightGBM] [Info] Number of positive: 271, number of negative: 7729
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000543 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2536
[LightGBM] [Info] Number of data points in the train set: 8000, number of used features: 12
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
Model trained successfully!


In [5]:
# Calculate metrics for all noise levels
noise_levels = [0, 0.05, 0.10, 0.15, 0.20]
noise_labels = ['0%', '5%', '10%', '15%', '20%']

print("=== Detailed Metrics Calculation ===\n")

all_metrics = []

for noise, label in zip(noise_levels, noise_labels):
    # Add noise
    np.random.seed(42)
    if noise == 0:
        X_test_noisy = X_test_scaled
    else:
        noise_data = np.random.normal(0, noise, X_test_scaled.shape)
        X_test_noisy = X_test_scaled + noise_data
    
    # Predictions
    pred = model.predict(X_test_noisy)
    proba = model.predict_proba(X_test_noisy)[:, 1]
    
    # Calculate all metrics
    metrics = {
        'noise_level': label,
        'accuracy': accuracy_score(Y_test, pred),
        'precision': precision_score(Y_test, pred, average='macro'),
        'recall': recall_score(Y_test, pred, average='macro'),
        'f1_score': f1_score(Y_test, pred, average='macro'),
        'roc_auc': roc_auc_score(Y_test, proba)
    }
    all_metrics.append(metrics)
    
    print(f"Noise Level: {label}")
    print(f"  Accuracy:  {metrics['accuracy']:.3f}")
    print(f"  Precision: {metrics['precision']:.3f}")
    print(f"  Recall:    {metrics['recall']:.3f}")
    print(f"  F1 Score:  {metrics['f1_score']:.3f}")
    print(f"  ROC-AUC:   {metrics['roc_auc']:.3f}")
    print()

=== Detailed Metrics Calculation ===

Noise Level: 0%
  Accuracy:  0.987
  Precision: 0.891
  Recall:    0.908
  F1 Score:  0.899
  ROC-AUC:   0.973

Noise Level: 5%
  Accuracy:  0.984
  Precision: 0.871
  Recall:    0.892
  F1 Score:  0.882
  ROC-AUC:   0.971

Noise Level: 10%
  Accuracy:  0.978
  Precision: 0.817
  Recall:    0.875
  F1 Score:  0.843
  ROC-AUC:   0.972

Noise Level: 15%
  Accuracy:  0.979
  Precision: 0.827
  Recall:    0.868
  F1 Score:  0.846
  ROC-AUC:   0.972

Noise Level: 20%
  Accuracy:  0.978
  Precision: 0.820
  Recall:    0.861
  F1 Score:  0.839
  ROC-AUC:   0.970



c:\Users\acer\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\acer\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\acer\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\acer\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\acer\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWa

In [6]:
metrics_df = pd.DataFrame(all_metrics)
print("=== Final Metrics Table ===")
print(metrics_df)

metrics_df.to_csv(
    r'C:\Infoct_project 1 sample practice\Week4_Yogesh\detailed_metrics.csv',
    index=False)
print("\nMetrics saved to CSV!")

=== Final Metrics Table ===
  noise_level  accuracy  precision    recall  f1_score   roc_auc
0          0%    0.9865   0.891256  0.907883  0.899381  0.973009
1          5%    0.9840   0.871369  0.892400  0.881569  0.971106
2         10%    0.9775   0.816819  0.874848  0.843151  0.971669
3         15%    0.9785   0.826749  0.868271  0.846147  0.971783
4         20%    0.9775   0.819995  0.860659  0.838991  0.969964

Metrics saved to CSV!
